In [128]:
import pandas as pd
import numpy as np

url = "https://raw.githubusercontent.com/justmarkham/DAT8/master/data/sms.tsv"
data = pd.read_csv(url, sep='\t', header=None, names=['label', 'message'],nrows=5100)
df = data.iloc[:5000,:]
test_data_x = data.iloc[5000:,1]
test_data_y = data["label"]
test_data_y = pd.get_dummies(test_data_y).astype(int)

test_data_y = test_data_y["spam"][5000:]

print(df.shape)
print(df.head())
np.bincount(test_data_y)

(5000, 2)
  label                                            message
0   ham  Go until jurong point, crazy.. Available only ...
1   ham                      Ok lar... Joking wif u oni...
2  spam  Free entry in 2 a wkly comp to win FA Cup fina...
3   ham  U dun say so early hor... U c already then say...
4   ham  Nah I don't think he goes to usf, he lives aro...


array([83, 17])

In [129]:
y = df["label"]
y

,label
0,ham
1,ham
2,spam
3,ham
4,ham
...,...
4995,ham
4996,ham
4997,ham
4998,ham


In [130]:
y = pd.get_dummies(y)
y = np.array(y)
y = y[:,1:]
y = np.concatenate(y)
y

array([False, False,  True, ..., False, False, False])

In [131]:
x = pd.DataFrame(df)
x = df["message"].str.lower().str.replace(r'[%©®™-✓*"\':؛!?+()/&_$#@~`|•√π÷×§∆\}{1234567890}{=°^¥€¢£%,]', ' ', regex=True).str.split()
x

,message
0,"[go, until, jurong, point, crazy.., available,..."
1,"[ok, lar..., joking, wif, u, oni...]"
2,"[free, entry, in, a, wkly, comp, to, win, fa, ..."
3,"[u, dun, say, so, early, hor..., u, c, already..."
4,"[nah, i, don, t, think, he, goes, to, usf, he,..."
...,...
4995,"[my, drive, can, only, be, read., i, need, to,..."
4996,"[just, looked, it, up, and, addie, goes, back,..."
4997,"[happy, new, year., hope, you, are, having, a,..."
4998,"[esplanade, lor., where, else...]"


In [132]:
x = np.array(x)
x_array = np.concatenate(x)
x_array = np.unique(x_array)
x_array = x_array[x_array != '-']

x_dict = {}
for l,i in enumerate(x_array) :
        x_dict[i] = l
x_dict
len(x_dict)

9594

In [133]:
zero_X = np.zeros((len(y), len(x_array)))


def encoder(zeros_array,text_massages) :

    for index,text in enumerate(text_massages) :
        for word in text :
            if word in x_dict :
                index_word = x_dict[word]
                zeros_array[index,index_word] = 1


    return zeros_array
X = encoder(zero_X,x)
X.shape

(5000, 9594)

In [134]:
ham,spam = np.bincount(y)
ham,spam

(np.int64(4327), np.int64(673))

In [135]:
x_spam = X[y==1]
x_ham = X[y==0]

repetition_ham = x_ham.sum(axis = 0)
repetition_spam = x_spam.sum(axis = 0)

w_ham = (1 + repetition_ham)/(x_ham.sum()+len(x_dict))
w_spam = (1 + repetition_spam)/(x_spam.sum()+len(x_dict))

In [136]:
test_data_x = test_data_x.str.lower().str.replace(r'[%©®™-✓*"\':؛!?+()/&_$#@~`|•√π÷×§∆\}{1234567890}{=°^¥€¢£%,]', ' ', regex=True).str.split()
massage = np.zeros((len(test_data_x), len(x_dict)))

test_data_x

,message
5000,"[hmph., go, head, big, baller.]"
5001,"[well, its, not, like, you, actually, called, ..."
5002,"[nope., since, ayo, travelled, he, has, forgot..."
5003,"[you, still, around, looking, to, pick, up, la..."
5004,"[cds, u, congratulations, ur, awarded, of, cd,..."
...,...
5095,"[k.k.this, month, kotees, birthday, know]"
5096,"[but, i, m, really, really, broke, oh., no, am..."
5097,"[sorry, about, that, this, is, my, mates, phon..."
5098,"[themob>hit, the, link, to, get, a, premium, p..."


In [137]:
my_test = encoder(massage,test_data_x)

spam_predictions = np.sum(my_test * np.log(w_spam), axis=1)

ham_predictions = np.sum(my_test * np.log(w_ham), axis=1)

spam_predictions,ham_predictions

(array([ -25.80931074,  -90.44166711,  -65.4012423 ,  -59.65905915,
        -136.16111917, -292.05156764, -105.28992042,  -34.36952751,
         -53.84517478,  -52.95839583,  -49.93578101,  -49.04792821,
        -138.9378861 , -100.68018688,  -89.42258253, -173.86887568,
        -123.47902991,  -47.66007847, -179.52752331, -141.87012529,
         -42.56922732, -103.15417475,  -10.08004251, -103.94123037,
        -147.1442211 ,  -34.13396143,  -81.74918564, -141.84255824,
         -97.72009688,  -96.74605404, -105.77147816, -118.08966268,
         -60.82676277,  -35.29198783,  -77.26643554,  -24.12951563,
        -103.78570059, -207.83574802,  -48.53579868, -156.71283797,
        -121.58418195, -129.80594775,  -98.95170155, -100.94073386,
         -62.71253506, -186.87274248, -126.64919503,  -88.61507038,
         -19.17661907, -125.49569435, -240.94259869,  -62.94083984,
         -69.79782234, -124.82730729,  -45.78287134, -133.47907597,
        -234.24611161,  -68.84857153, -160.13698

In [139]:
final_predictions = np.where(spam_predictions > ham_predictions, 1, 0)

test_data_y = np.array(test_data_y)

np.bincount(final_predictions)

array([81, 19])